# The reduction, one stage at a time

`nirc2pol-reduce night.toml` calls `nirc2pol.polmode.run`, which does
everything below in one go. This notebook is that function opened up: the
same ten stages, in the same order, reading the same config.

**It is meant to stay faithful, not to be a simplified version.** Every stage
takes its settings from `cfg`, including the ones that are easy to leave out
by hand — the seven `reduce_frame` arguments that decide how flats are
matched and skies chosen, the master-building overrides, the leakage. A
notebook that omits one of those does not fail; it quietly reduces differently
from the pipeline, and the difference turns up later as a result you cannot
reproduce.

Verified against `run` on two nights that exercise different branches:
undithered with no leakage correction, and dithered with `fit_uphi_all`. Both
come out **bit-identical**.

Change the config in section 0 and every stage follows, exactly as the CLI
would.

In [ ]:
import logging, os

import matplotlib.pyplot as plt
import numpy as np

from nirc2pol.instruments import nirc2
from nirc2pol.instruments.nirc2 import NIRC2PolarimetryData
from nirc2pol.polarimetry import (ProductWriter, apply_mueller_model,
                                  aperture_polarization, build_stokes_cubes,
                                  fit_ip_uphi, fit_ip_uphi_all, mean_ip,
                                  fit_fast_axis_butterfly, median_stokes_cube,
                                  radial_stokes)
from nirc2pol.reduction import (background_stages, make_master_darks,
                                make_master_flats, make_master_masks,
                                make_master_skies, reduce_frame,
                                subtract_dither_background)
from nirc2pol.reduction.config import ReductionConfig
from nirc2pol.utils import (ObslogPaths, load_frames, load_rejects,
                            read_headers, save_frames, select_frames,
                            start_reduction_log)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("step_by_step")

## 0. The config

Everything the reduction decides is here. `nirc2pol-reduce --template >
night.toml` prints the same fields with their documentation, and
`ReductionConfig.from_toml(path)` reads one back — use that instead of the
literal below to reduce a night you have already described.

The defaults point at an undithered L′ night with no leakage correction.
The commented block underneath is a dithered night with the leakage fitted,
which exercises the other branches of stages 6 and 8.

In [ ]:
cfg = ReductionConfig(
    raw_data_folder="/home/shared/exoserver/NIRC2_Pol/20251207",
    reductions_root="/home/blewis/reductions/ab_aur_Lp_20251207",
    date="2025-12-07",
    target="AB_Aur",

    raw_range=[857, 993],                     # what is read off disk at all
    select_frame_range=[[857, 900], [915, 930], [932, 963]],   # the science

    background_method="mean_box",             # not dithered
    background_box=[25, 350, 50, 400],

    register_method="min",                    # this core is saturated
    fast_axis_method="fixed",
    theta_off=-13.1,
    ip_method=None,
    ip_mask_radius=22.0,

    save_preproc=True,
    save_individual_cycles=False,
)

# A dithered night with the leakage fitted:
#
# cfg = ReductionConfig(
#     raw_data_folder="/home/shared/exoserver/NIRC2_Pol/20251206",
#     reductions_root="/tmp/io_step", date="2025-12-06", target="Io",
#     raw_range=[171, 617], select_frame_range=[[487, 498], [500, 571]],
#     background_method=["dither"],   # or ["dither", "annulus"]
#     register_method="centroid",     # Io is a resolved disk
#     fast_axis_method="fixed", theta_off=-13.1,
#     ip_method="fit_uphi_all", ip_mask_radius=0.0,
#     save_preproc=True, save_individual_cycles=False,
# )

# ...or read one you already have:
# cfg = ReductionConfig.from_toml("night.toml")

## Setup: instrument, paths, links, rejects, log

`configure` copies the config's choices onto the instrument. The frames are
**symlinked** into `reductions_root/raw` rather than read in place, so the
archive is never written to and the run records exactly which files it saw.
Nothing is created until the frames are known to exist, so a wrong
`raw_data_folder` leaves no empty tree behind.

In [ ]:
instrument = cfg.configure(NIRC2PolarimetryData())

paths = ObslogPaths(cfg.reductions_root, cfg.date)
raw_files = paths.link_raw_frames(cfg.raw_data_folder,
                                  frame_range=cfg.raw_range)
paths.make_folders()

# Frames excluded from every run of this night, each with a reason:
#     from nirc2pol.utils.paths import record_reject
#     record_reject(paths.rejects_file, "n0937.fits", "open AO loop")
rejects = load_rejects(paths.rejects_file)

# The config is copied in beside the log: the log records the values, but a
# file can be re-run, and the one passed on the command line may be edited or
# gone by the time anyone comes back to this.
saved_config = cfg.to_toml(paths.config_file)

run_log = start_reduction_log(paths.log_file)
run_log.settings(instrument=type(instrument).__name__,
                 background=instrument.describe_background(),
                 config=saved_config, **cfg.describe())
log.info("background: %s", instrument.describe_background())
print(f"{len(raw_files)} raw files linked into {paths.raw_folder}")
print(f"{len(rejects)} rejected frame(s); log: {run_log.path}")

## 1. Sort the raw frames by type

`raw_range` decided what is read off disk at all, which is a different
question from `select_frame_range` — that picks the science frames later.
**`raw_range` has to stay wide enough to include the darks and flats**, or the
masters cannot be built.

In [ ]:
sorted_files = instrument.sort_frames(raw_files)
for kind, files in sorted_files.items():
    print(f"   {kind:12s} {len(files)}")

## 2. Master darks, flats and skies

`instrument=` supplies the detector mask, how to spot a polarimetric
(critical-angle) flat set, and which flat type each band requires. The `cfg`
arguments override those per run — leaving any of them out is one of the ways
a hand-written reduction silently stops matching the pipeline.

`science_bands` is ordering only: the flats in the science bands lead the
inventory, but `find_closest_flat` still requires the filter to match, so it
cannot change which flat a frame gets.

`save_preproc` decides whether the masters are kept on disk. They are built
either way — nothing downstream runs without them — so it is about disk, not
about what the reduction does.

In [ ]:
darks = load_frames(sorted_files["darks"], rejects=rejects)
master_darks, dark_masks = make_master_darks(
    darks, instrument=instrument, min_frames=cfg.master_min_frames)
if master_darks and cfg.save_preproc:
    save_frames(paths.darks_file, master_darks)

# Headers only -- the science frames are not loaded until stage 3.
science_bands = {nirc2.band_of(h)
                 for h in read_headers(sorted_files["sci"])}
science_bands.discard(None)

master_flats, flat_masks = make_master_flats(
    load_frames(sorted_files["flats_dome"], rejects=rejects),
    load_frames(sorted_files["flats_sky"], rejects=rejects),
    master_darks,
    instrument=instrument,
    required_flat_type=cfg.required_flat_type,
    allow_flat_without_dark=cfg.allow_flat_without_dark,
    min_frames=cfg.master_min_frames,
    science_bands=science_bands,
)
if master_flats and cfg.save_preproc:
    save_frames(paths.flats_file, master_flats)

master_skies = None
if cfg.use_master_skies:
    master_skies, _ = make_master_skies(
        load_frames(sorted_files["flats_sky"], rejects=rejects),
        master_darks, instrument=instrument,
        min_frames=cfg.master_min_frames,
        group_radius_arcsec=cfg.sky_group_radius,
        group_gap_minutes=cfg.sky_group_gap)
    if master_skies and cfg.save_preproc:
        save_frames(paths.skies_file, master_skies)

master_masks = make_master_masks(dark_masks, flat_masks)

print(f"{len(master_darks)} master darks, {len(master_flats)} master flats")
for f in master_flats:
    print(f"   {f['FILTER']:16s} FLATTYPE {f['FLATTYPE']:5s} "
          f"polarimetric {f.get('POLFLAT')}  from {f['NFRAMES']} frames")

## 3. Choose the frames this reduction covers

**Before reducing, not after.** `sort_frames` classifies by elimination —
anything that is not a dark or a flat is science — so the science bucket also
holds acquisition and engineering frames, in whatever band they were taken.
Reducing the whole bucket first spends the time on frames about to be
discarded, and fails outright on one in a band this night has no flat for.

The frame table is written for the **whole** night, since it is the thing you
read while deciding what to select.

In [ ]:
sci_frames = load_frames(sorted_files["sci"], rejects=rejects)
paths.check_frame_dates(sci_frames)        # folder date vs DATE-OBS
nirc2.make_frametable(sci_frames, paths.table_file)

sci_frames = select_frames(sci_frames, target=cfg.select_target,
                           frame_range=cfg.select_frame_range)
print(f"{len(sci_frames)} frames selected")

## 4. Pre-process them

Dark, flat, bad pixels — per frame, nothing polarimetric yet.

**The seven `cfg` arguments here are the ones most often dropped by hand.**
They decide whether a flat of the wrong kind is accepted, whether a frame with
no flat at all proceeds, which sky is chosen and how far away it may be, and
how bad pixels are replaced. Omitting them does not error; it reduces
differently.

`required_flat_type` is passed the same value the masters were built with:
choosing the kind for the build and then enforcing the band default when
matching would refuse the flats it had just made.

In [ ]:
bad_pixel_mask = instrument.bad_pixel_mask()   # loads a FITS; read once

reduced_frames = []
for frame in sci_frames:
    reduced = reduce_frame(
        frame, master_flats, master_darks, master_skies, master_masks,
        bad_pixel_mask=bad_pixel_mask,
        required_flat_types=instrument.required_flat_types,
        default_required_flat_type=instrument.default_required_flat_type,
        required_flat_type=cfg.required_flat_type,
        allow_flat_type_mismatch=cfg.allow_flat_type_mismatch,
        allow_no_flat=cfg.allow_no_flat,
        skip_sky_sub=cfg.skip_sky_sub,
        sky_group_radius_arcsec=cfg.sky_group_radius,
        sky_max_radius_arcsec=cfg.sky_max_radius,
        replacement_method=cfg.replacement_method,
        gain=instrument.gain(frame),
        saturation_limit=instrument.saturation_limit(frame),
    )
    reduced_frames.append(reduced)

print(f"{len(reduced_frames)} frames reduced")

## 5. The beam cutout

Where `split_beams` cuts the two beams out. **Nominal per-band values, not a
measurement** — they only have to contain each beam. Whatever offset they
leave between the beams is removed per frame by `align_beams` during
registration, which is simpler and more accurate than choosing this pair well:
the two beams are rotated ~0.37° relative to each other, so their separation
depends on where the source sits in the field and no single pair is right at
more than one position.

Keyed on the **config**, not on the instrument: `configure` leaves the
instrument's nominal values in place rather than nulling them, so testing the
instrument would never fire and the band-specific entry would never be reached.

In [ ]:
if cfg.beam_top_row is None or cfg.beam_x_offset is None:
    band = nirc2.band_of(reduced_frames[0]) if reduced_frames else None
    nominal_top, nominal_x = type(instrument).beam_geometry_for(band)
    if cfg.beam_top_row is None:
        instrument.top_row_start = nominal_top
    if cfg.beam_x_offset is None:
        instrument.beam_x_offset = nominal_x

print(f"beam cutout: top row {instrument.top_row_start}, "
      f"x offset {instrument.beam_x_offset} (nominal; align_beams removes "
      f"the residual per frame)")

## 6. Dither subtraction

At **frame** level, before the Wollaston beams are cut out, and pair-matched
within one HWP angle so it differences two skies rather than two polarization
states. Any other stage in the chain — `annulus`, `mean_box` — runs later and
per beam, inside the Stokes builder, on what this leaves behind.

`build_stokes_cubes` would apply this itself if it were skipped, so a notebook
that forgets it still gets a background subtracted. It is written out here
because this is where `run` does it, and because the reduced frames saved
below should be what the Stokes cubes were actually built from.

Saved **after** the dither, not before: `DITHSUB` records which frame was
subtracted from each.

In [ ]:
if "dither" in background_stages(cfg.background_method):
    reduced_frames = subtract_dither_background(
        reduced_frames, instrument, tolerance_arcsec=cfg.dither_tolerance)

if cfg.save_preproc:
    for reduced in reduced_frames:
        reduced.save(os.path.join(paths.reduced_folder, reduced["RED-FN"]))

print(f"background: {instrument.describe_background()}")
print(f"DITHSUB on {sum(1 for f in reduced_frames if f.get('DITHSUB'))} "
      f"of {len(reduced_frames)} frames")

## 7. HWP cycle matching

The half-wave plate steps through the four critical angles 0°, 45°, 22.5°,
67.5° (`PCUPR`). `match_modulator_cycles` walks the frames in time order and
groups them into complete cycles; incomplete trailing groups are dropped with
a warning.

In [ ]:
cycles = instrument.match_modulator_cycles(reduced_frames)
print(f"{len(cycles)} complete HWP cycles")
print("cycle 0 HWP angles:",
      [round(instrument.modulator_angle(f), 1) for f in cycles[0]])

## 8. Fast axis offset, then the instrumental polarization

Both are chosen by naming a method, and **the order matters**: every available
IP route needs the offset as an input, so the leakage cannot be measured
first. When the offset is fitted rather than fixed, it is therefore fitted
with the leakage still in, and is biased by however much of it there is —
`run` warns about exactly that, and so does this.

`ip_method` is the setting whose omission produced the symptom that sent most
people here: a DoLP map pinned at ~1% everywhere with no structure, because
NIRC2's I → Q/U leakage is 1–2% and swamps a source polarized at a few tenths
of a percent.

In [ ]:
theta_off, ip = None, None
dd_kwargs = {"register_method": cfg.register_method}

if cfg.fast_axis_method == "mm_model":
    # Applies a matrix determined elsewhere; raises until that plumbing exists
    theta_off, ip = apply_mueller_model(instrument, cycles)

if theta_off is None:
    if cfg.fast_axis_method == "fixed":
        theta_off = cfg.theta_off
        log.info("fast axis: fixed at %g deg (nothing fitted)", theta_off)
    elif cfg.fast_axis_method == "butterfly":
        r_inner, r_outer = cfg.fit_radii
        result = fit_fast_axis_butterfly(instrument, cycles, ip=None,
                                         r_inner=r_inner, r_outer=r_outer,
                                         **dd_kwargs)
        theta_off = result.theta_off
        log.info("fast axis from the butterfly: %s", result.describe())
        if cfg.ip_method is not None:
            log.warning("Offset fitted with no leakage removed -- the two are "
                        "degenerate, so this offset carries the IP.")

if cfg.ip_method == "fit_uphi_all":
    ip = fit_ip_uphi_all(instrument, cycles, theta_off,
                         mask_radius=cfg.ip_mask_radius, **dd_kwargs)
elif cfg.ip_method == "fit_uphi_per_cycle":
    ip = [fit_ip_uphi(instrument, c, theta_off,
                      mask_radius=cfg.ip_mask_radius, **dd_kwargs)
          for c in cycles]

if isinstance(ip, list):
    log.info("instrumental polarization, per cycle (mean %s)",
             mean_ip(ip).describe())
elif ip is not None:
    log.info("instrumental polarization: %s", ip.describe())

instrument.fast_axis_offset = theta_off
print(f"theta_off = {theta_off} deg, ip_method = {cfg.ip_method}")

## 9. Stokes cubes

Beam splitting, alignment, per-beam background, registration and the double
difference all happen inside the builder, once per cycle.

The leakage is removed **in the instrument frame, before Q/U are rotated to
sky**, which is why it is an argument here rather than something subtracted
from a finished cube.

`crop_size` trims the negative dither ghost. Its size is derived from the
throw and the source radius, so `None` is the normal setting; nothing is
cropped when the data was not dithered.

In [ ]:
stokes_cubes = build_stokes_cubes(instrument, cycles,
                                  fast_axis_offset=theta_off, ip=ip,
                                  crop_size=cfg.crop_size, **dd_kwargs)
median_cube = median_stokes_cube(stokes_cubes)
print(f"{len(stokes_cubes)} cycle cubes -> median {median_cube.shape}")

## 10. Products

`THETAOFF` is already on the header — `build_stokes_cubes` wrote it when it
resolved the offset actually used, so nothing needs to stamp it on by hand.

`save_individual_cycles` decides whether the per-cycle data is kept: one FITS
per cycle, each with its own header. Off, the median cube and the derived
products are all that is written, and no cycle can be dropped or re-combined
without reducing the night again.

In [ ]:
header = cycles[0][0].header.copy()

writer = ProductWriter(paths.sequences_folder, target=cfg.target,
                       overwrite=cfg.overwrite_products)

if cfg.save_individual_cycles:
    writer.save_stokes_cycles(stokes_cubes, cycles, header=header)
writer.save_median_stokes(median_cube, header=header)
writer.save_derived_products(median_cube, header=header,
                             derived=cfg.save_derived_quantities,
                             radial=cfg.save_radial_stokes,
                             dolp_min_intensity=cfg.dolp_min_intensity)

run_log.finish()
print(f"\nproducts in {writer.output_dir}")
print(f"log: {run_log.path} ({run_log.warnings} warnings)")

## Look at it

`U_phi` is the null channel: a tangentially polarized source puts nothing
there, so what is in it measures what the reduction got wrong.

`aperture_polarization` gives integrated `p` and position angle. Give it a
background annulus — Q and U are differences, so the sky is expected to cancel
and does not quite — and read the numbers as a series, because a `p` that
moves with radius is telling you about the background rather than the source.

In [ ]:
qphi, uphi = radial_stokes(median_cube[1], median_cube[2])

v = np.nanpercentile(np.abs(uphi[np.isfinite(uphi)]), 99.6)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for a_, img, title in [(ax[0], qphi, "$Q_\\phi$"),
                       (ax[1], uphi, "$U_\\phi$ (null channel)")]:
    im = a_.imshow(img, origin="lower", vmin=-v, vmax=3 * v, cmap="inferno")
    a_.set_title(title, fontsize=12)
    fig.colorbar(im, ax=a_, fraction=0.046)
fig.suptitle(f"{cfg.target} {cfg.date}, theta_off = {theta_off} deg, "
             f"ip_method = {cfg.ip_method}", fontsize=13)
fig.tight_layout()

finite = np.isfinite(uphi) & np.isfinite(qphi)
print(f"Q_phi max {np.nanmax(qphi):.1f},  "
      f"U_phi std {np.nanstd(uphi[finite]):.3f}")

## Does this match the pipeline?

The point of the notebook is that it should. Reducing the same config both
ways and differencing the Stokes cubes is the check, and it is worth running
whenever a stage above has been edited.

In [ ]:
# from nirc2pol.polmode import run
#
# reference = run(ReductionConfig.from_toml(paths.config_file))
# delta = np.nanmax(np.abs(np.nan_to_num(median_cube)
#                          - np.nan_to_num(reference["median_cube"])))
# print(f"max abs difference from polmode.run: {delta:g}")